# Masked Inference — Mechatronics Vision 2026

**Phase 2.** Depth-guided YOLO inference on the ZED.

1. Grab an RGB frame + depth matrix from the ZED.
2. Downsample the depth matrix, build a 3D point cloud, cluster it with DBSCAN.
3. Drop DBSCAN clusters whose median depth is beyond `WALL_DEPTH_M` (back walls).
4. Crop the RGB frame to each kept cluster's 2D bbox and run YOLO on that crop only.
5. Remap detections back to the full frame and draw them with depth readouts.

Shutter modes: `single` / `burst` / `continuous`. See Cell 2 for every knob.


## 1. Imports

In [ ]:
import sys, time
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import cv2
import pyzed.sl as sl
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler
from ultralytics import YOLO

# plotly is only needed for the optional 3D viz cell at the bottom
PROJECT_ROOT = Path('/Users/jnoael/Mechatronics_Vision_2026')
sys.path.insert(0, str(PROJECT_ROOT))
print('imports OK')


## 2. Config

In [ ]:
# ── Model ─────────────────────────────────────────────────────────
WEIGHTS    = PROJECT_ROOT / 'runs_cnn_transfer' / 'v1.1' / 'weights' / 'best.pt'
CONF_THRES = 0.50
IOU_THRES  = 0.50
IMGSZ      = 640     # YOLO input size (crops are resized to this)
MAX_DET    = 4       # top-N most confident detections per cluster crop
CLASSES    = None    # None = all classes, or list of class-id ints e.g. [0, 3]
AUGMENT    = False   # YOLO test-time augmentation (slower, slightly better)
DEVICE     = None    # None = auto; 0 for cuda:0, 'mps' for Apple, 'cpu' to force

# ── ZED camera ────────────────────────────────────────────────────
RESOLUTION = sl.RESOLUTION.HD720              # VGA | HD720 | HD1080 | HD2K
FPS        = 30
DEPTH_MODE = sl.DEPTH_MODE.PERFORMANCE        # PERFORMANCE | QUALITY | NEURAL
DEPTH_MIN_M = 0.3                             # ignore points closer than this

# ── Shutter ───────────────────────────────────────────────────────
SHUTTER_MODE          = 'continuous'  # 'single' | 'burst' | 'continuous'
BURST_COUNT           = 5             # frames captured back-to-back for 'burst'
CONTINUOUS_TARGET_FPS = 10            # target inference FPS for 'continuous'

# ── Depth -> point cloud ──────────────────────────────────────────
# Depth compression: keep every Nth pixel in X and Y before building the
# point cloud that feeds DBSCAN. HD720 at step=4 -> ~57k points (vs 920k raw),
# which is the biggest single speed lever for the whole pipeline.
DEPTH_DOWNSAMPLE_STEP = 4

# ── DBSCAN ────────────────────────────────────────────────────────
# Tune eps first. Both are applied in [0,1]-normalised space so they're
# insensitive to resolution changes.
DBSCAN_EPS         = 0.03
DBSCAN_MIN_SAMPLES = 20
MIN_CLUSTER_POINTS = 50   # drop clusters with fewer points than this

# ── Wall filter ───────────────────────────────────────────────────
# Any cluster whose MEDIAN point depth exceeds this is assumed to be the
# back wall / distant geometry and dropped before YOLO runs.
WALL_DEPTH_M = 5.0

# ── Continuous-mode performance ───────────────────────────────────
# DBSCAN is the expensive step; re-run it every Nth frame and reuse the
# last cluster bboxes for YOLO on intermediate frames.
DBSCAN_EVERY_N_FRAMES   = 3
MAX_CLUSTERS_PER_FRAME  = 6     # cap YOLO calls per frame by cluster count

# ── Display / output ──────────────────────────────────────────────
# 'window'    : live cv2.imshow preview (continuous / single / burst all support this)
# 'save_only' : headless, write annotated frames to SAVE_DIR (good over SSH)
DISPLAY_MODE = 'window'
SAVE_DIR     = PROJECT_ROOT / 'runs_inference'

# ── 3D DBSCAN visualisation (bottom cell) ─────────────────────────
# When True, the last cell renders a plotly 3D scatter of the most recent
# pipeline run's point cloud, colouring kept clusters vs dropped walls vs
# noise. Debug-only -- leave False for production.
VISUALIZE_CLUSTERS_3D = False

print(f'weights      : {WEIGHTS}')
print(f'shutter mode : {SHUTTER_MODE}')
print(f'display mode : {DISPLAY_MODE}')


## 3.1 Capture — frame + depth matrix

`open_zed()` configures the camera from the constants above. `capture_frame_and_depth()` grabs one synced pair: a BGR image and the Z-depth matrix (float32 meters; NaN / inf where depth is unknown).


In [ ]:
def open_zed() -> sl.Camera:
    zed = sl.Camera()
    init = sl.InitParameters()
    init.camera_resolution      = RESOLUTION
    init.camera_fps             = FPS
    init.depth_mode             = DEPTH_MODE
    init.coordinate_units       = sl.UNIT.METER
    init.depth_minimum_distance = DEPTH_MIN_M
    err = zed.open(init)
    if err != sl.ERROR_CODE.SUCCESS:
        raise RuntimeError(f'ZED open failed: {err}')
    cam_info = zed.get_camera_information()
    res = cam_info.camera_configuration.resolution
    print(f'ZED opened  {res.width}x{res.height} @ {FPS} FPS  depth={DEPTH_MODE}')
    return zed


def capture_frame_and_depth(zed, runtime, image_mat, depth_mat
                            ) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """Grab one synced frame. Returns (rgb_bgr HxWx3, depth_m HxW meters).
    Returns (None, None) on a bad grab -- caller should retry."""
    if zed.grab(runtime) != sl.ERROR_CODE.SUCCESS:
        return None, None
    zed.retrieve_image(image_mat,  sl.VIEW.LEFT)
    zed.retrieve_measure(depth_mat, sl.MEASURE.DEPTH)  # Z-depth along optical axis

    rgb = image_mat.get_data()
    if rgb.ndim == 3 and rgb.shape[2] == 4:
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGRA2BGR)
    rgb   = np.ascontiguousarray(rgb)
    depth = np.ascontiguousarray(depth_mat.get_data())
    return rgb, depth


## 3.2 Depth matrix → DBSCAN clusters

Downsamples the depth matrix by `DEPTH_DOWNSAMPLE_STEP`, drops invalid depths, builds a `(x_pixel, y_pixel, depth_m)` point cloud, [0,1]-normalises each axis so DBSCAN's isotropic `eps` is fair across mixed units, and clusters. Returns the raw full-resolution points alongside the cluster labels so the next step can work in pixel space.


In [ ]:
def cluster_depth(depth_m: np.ndarray,
                   step: int = DEPTH_DOWNSAMPLE_STEP,
                   eps: float = DBSCAN_EPS,
                   min_samples: int = DBSCAN_MIN_SAMPLES
                   ) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (points_fullres, labels) where points_fullres columns are
    [x_full_pixel, y_full_pixel, depth_meters] and labels is int array with -1=noise."""
    small  = depth_m[::step, ::step]
    h, w   = small.shape
    yy, xx = np.mgrid[0:h, 0:w]

    flat = small.ravel()
    ok   = np.isfinite(flat) & (flat > 0)
    if ok.sum() < min_samples:
        return np.empty((0, 3), dtype=np.float32), np.empty((0,), dtype=np.int64)

    points = np.column_stack([
        xx.ravel()[ok] * step,   # back to full-resolution X
        yy.ravel()[ok] * step,   # back to full-resolution Y
        flat[ok],                # depth in meters
    ]).astype(np.float32)

    points_norm = MinMaxScaler().fit_transform(points)
    labels = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(points_norm)
    return points, labels


def filter_clusters(points: np.ndarray, labels: np.ndarray,
                    wall_depth_m: float = WALL_DEPTH_M,
                    min_points: int = MIN_CLUSTER_POINTS,
                    max_clusters: int = MAX_CLUSTERS_PER_FRAME
                    ) -> List[dict]:
    """Drop noise, drop tiny clusters, drop distant-wall clusters. Keep the
    `max_clusters` largest remaining. Returns dicts with 2D bbox + median depth."""
    kept = []
    for lbl in set(labels.tolist()):
        if lbl == -1:
            continue
        mask = labels == lbl
        n    = int(mask.sum())
        if n < min_points:
            continue
        xs, ys, zs = points[mask, 0], points[mask, 1], points[mask, 2]
        median_depth = float(np.median(zs))
        if median_depth > wall_depth_m:
            continue
        kept.append(dict(
            label        = int(lbl),
            bbox         = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
            median_depth = median_depth,
            n_points     = n,
        ))
    # largest first; cap to MAX_CLUSTERS_PER_FRAME
    kept.sort(key=lambda c: c['n_points'], reverse=True)
    return kept[:max_clusters]


## 3.3 Masked YOLO — run only on cluster crops

For each kept cluster, the RGB frame is cropped to the cluster's 2D bbox (with a few pixels of padding so YOLO has context), YOLO runs on that crop, and any detections are remapped to full-frame coordinates. Depth from the source cluster is attached.


In [ ]:
PALETTE = [(0, 255, 0), (0, 255, 255), (255, 128, 0), (255, 0, 255),
           (0, 128, 255), (128, 255, 0)]


def infer_on_clusters(model: YOLO, rgb: np.ndarray, clusters: List[dict],
                      pad: int = 8) -> List[dict]:
    """Runs YOLO on each cluster crop independently. Detections are returned
    in full-frame coordinates with their source cluster's depth attached."""
    detections = []
    H, W = rgb.shape[:2]
    device_kw = {} if DEVICE is None else {'device': DEVICE}
    for cl in clusters:
        x1, y1, x2, y2 = cl['bbox']
        x1 = max(0, x1 - pad); y1 = max(0, y1 - pad)
        x2 = min(W, x2 + pad); y2 = min(H, y2 + pad)
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        crop = rgb[y1:y2, x1:x2]

        r = model.predict(crop, conf=CONF_THRES, iou=IOU_THRES,
                          imgsz=IMGSZ, max_det=MAX_DET,
                          classes=CLASSES, augment=AUGMENT,
                          verbose=False, **device_kw)[0]
        if r.boxes is None or len(r.boxes) == 0:
            continue
        xyxy   = r.boxes.xyxy.cpu().numpy()
        confs  = r.boxes.conf.cpu().numpy()
        clsids = r.boxes.cls.cpu().numpy().astype(int)
        for bb, cf, k in zip(xyxy, confs, clsids):
            gx1, gy1, gx2, gy2 = bb + np.array([x1, y1, x1, y1])
            detections.append(dict(
                bbox           = (int(gx1), int(gy1), int(gx2), int(gy2)),
                cls            = int(k),
                cls_name       = model.names.get(int(k), str(k)),
                conf           = float(cf),
                cluster_depth  = cl['median_depth'],
            ))
    return detections


def annotate(rgb: np.ndarray, clusters: List[dict], detections: List[dict],
              fps: Optional[float] = None) -> np.ndarray:
    """Draw cluster bboxes (faint) + YOLO detections (bold, coloured)."""
    out = rgb.copy()
    for cl in clusters:
        x1, y1, x2, y2 = cl['bbox']
        cv2.rectangle(out, (x1, y1), (x2, y2), (200, 200, 200), 1)
        cv2.putText(out, f'z={cl["median_depth"]:.1f}m',
                    (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                    (200, 200, 200), 1, cv2.LINE_AA)
    for d in detections:
        x1, y1, x2, y2 = d['bbox']
        color = PALETTE[d['cls'] % len(PALETTE)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        tag = f"{d['cls_name']} {d['conf']:.2f} | {d['cluster_depth']:.2f}m"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(out, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, -1)
        cv2.putText(out, tag, (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2, cv2.LINE_AA)
    if fps is not None:
        cv2.putText(out, f'{fps:5.1f} FPS  |  {len(detections)} det',
                    (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    (255, 255, 255), 2, cv2.LINE_AA)
    return out


## 4. Pipeline orchestrator — single / burst / continuous

`run_pipeline()` reads `SHUTTER_MODE` from config and dispatches to the matching loop.

- `single`     — grab one frame, run the full pipeline, show/save, exit.
- `burst`      — grab `BURST_COUNT` frames as fast as the camera allows, then run
                 the pipeline on each (capture-then-process so frame timing stays tight).
- `continuous` — loop at ~`CONTINUOUS_TARGET_FPS`, re-running DBSCAN every
                 `DBSCAN_EVERY_N_FRAMES` frames and YOLO every frame. Press `q` to quit.

The last processed frame's intermediates are stashed in `_last_results` so the 3D viz
cell at the bottom can render them on demand.


In [ ]:
_last_results: dict = {}


def _grab(zed, runtime, image_mat, depth_mat, retries: int = 30):
    for _ in range(retries):
        rgb, depth = capture_frame_and_depth(zed, runtime, image_mat, depth_mat)
        if rgb is not None and depth is not None:
            return rgb, depth
    raise RuntimeError('ZED stopped providing frames')


def _present_or_save(annotated: np.ndarray, tag: str) -> bool:
    """Returns True if the caller should continue; False if the user quit."""
    if DISPLAY_MODE == 'window':
        cv2.imshow('masked inference (press q to quit, any key to continue)', annotated)
        key = cv2.waitKey(0) & 0xFF
        return key != ord('q')
    elif DISPLAY_MODE == 'save_only':
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        path = SAVE_DIR / f'{tag}.png'
        cv2.imwrite(str(path), annotated)
        print(f'saved {path}')
        return True
    else:
        raise ValueError(f'unknown DISPLAY_MODE: {DISPLAY_MODE!r}')


def _stash(**kv):
    _last_results.update(kv)


def _process_frame(model, rgb, depth):
    points, labels = cluster_depth(depth)
    clusters       = filter_clusters(points, labels)
    detections     = infer_on_clusters(model, rgb, clusters)
    _stash(rgb=rgb, depth=depth, points=points, labels=labels,
           clusters=clusters, detections=detections)
    return clusters, detections


def _run_single(model, zed, runtime, image_mat, depth_mat):
    rgb, depth = _grab(zed, runtime, image_mat, depth_mat)
    clusters, detections = _process_frame(model, rgb, depth)
    print(f'single: {len(clusters)} cluster(s), {len(detections)} detection(s)')
    _present_or_save(annotate(rgb, clusters, detections), 'single')


def _run_burst(model, zed, runtime, image_mat, depth_mat):
    frames = []
    print(f'burst: capturing {BURST_COUNT} frame(s)...')
    for _ in range(BURST_COUNT):
        frames.append(_grab(zed, runtime, image_mat, depth_mat))
    print(f'burst: processing {len(frames)} frame(s)...')
    for i, (rgb, depth) in enumerate(frames):
        clusters, detections = _process_frame(model, rgb, depth)
        annotated = annotate(rgb, clusters, detections)
        if not _present_or_save(annotated, f'burst_{i:03d}'):
            break


def _run_continuous(model, zed, runtime, image_mat, depth_mat):
    print(f'continuous: target {CONTINUOUS_TARGET_FPS} FPS, '
          f'DBSCAN every {DBSCAN_EVERY_N_FRAMES} frames. Press q to quit.')
    frame_period = 1.0 / max(CONTINUOUS_TARGET_FPS, 1e-3)
    last_clusters: List[dict] = []
    frame_i, t0 = 0, time.time()
    while True:
        t_loop = time.time()
        rgb, depth = _grab(zed, runtime, image_mat, depth_mat)

        if frame_i % max(DBSCAN_EVERY_N_FRAMES, 1) == 0:
            points, labels = cluster_depth(depth)
            last_clusters  = filter_clusters(points, labels)
            _stash(points=points, labels=labels, clusters=last_clusters)

        detections = infer_on_clusters(model, rgb, last_clusters)
        _stash(rgb=rgb, depth=depth, detections=detections)

        fps = (frame_i + 1) / max(time.time() - t0, 1e-6)
        annotated = annotate(rgb, last_clusters, detections, fps=fps)

        if DISPLAY_MODE == 'window':
            cv2.imshow('masked inference (press q to quit)', annotated)
            if (cv2.waitKey(1) & 0xFF) == ord('q'):
                break
        elif DISPLAY_MODE == 'save_only':
            SAVE_DIR.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(SAVE_DIR / f'cont_{frame_i:06d}.png'), annotated)

        frame_i += 1
        elapsed = time.time() - t_loop
        if elapsed < frame_period:
            time.sleep(frame_period - elapsed)
    total = time.time() - t0
    print(f'continuous: stopped after {frame_i} frames in {total:.1f}s '
          f'({frame_i / max(total, 1e-6):.1f} FPS avg)')


def run_pipeline():
    if not WEIGHTS.exists():
        raise FileNotFoundError(f'weights not found: {WEIGHTS}')
    print(f'loading {WEIGHTS}...')
    model = YOLO(str(WEIGHTS))
    print(f'classes: {model.names}')

    zed = open_zed()
    runtime   = sl.RuntimeParameters()
    image_mat = sl.Mat()
    depth_mat = sl.Mat()
    try:
        {
            'single'     : _run_single,
            'burst'      : _run_burst,
            'continuous' : _run_continuous,
        }[SHUTTER_MODE](model, zed, runtime, image_mat, depth_mat)
    except KeyError:
        raise ValueError(f'unknown SHUTTER_MODE: {SHUTTER_MODE!r}')
    finally:
        zed.close()
        cv2.destroyAllWindows()
        print('ZED closed.')


## 5. Run the pipeline

In [ ]:
run_pipeline()


## 6. (Optional) 3D DBSCAN visualisation

Plotly 3D scatter of the most recently processed frame's point cloud. Kept clusters show in colour, dropped wall / noise points in grey. Use this to sanity-check `DBSCAN_EPS`, `DBSCAN_MIN_SAMPLES`, and `WALL_DEPTH_M` — change them in Cell 2, re-run the pipeline, then re-run this cell.

Gated by `VISUALIZE_CLUSTERS_3D` in the config cell so continuous runs don't stall.


In [ ]:
if not VISUALIZE_CLUSTERS_3D:
    print('VISUALIZE_CLUSTERS_3D is False. Set it True in Cell 2 and re-run the pipeline '
          '(then re-run this cell) to see the 3D scatter.')
elif not _last_results:
    print('No pipeline output stashed yet -- run Cell 5 first.')
else:
    import plotly.graph_objects as go
    points   = _last_results.get('points')
    labels   = _last_results.get('labels')
    clusters = _last_results.get('clusters', [])
    if points is None or labels is None or len(points) == 0:
        print('No point cloud in last run (was the frame empty?).')
    else:
        pts = MinMaxScaler().fit_transform(points)
        PALETTE_3D = ['#4C6EF5', '#F03E3E', '#37B24D', '#F59F00',
                      '#AE3EC9', '#1098AD', '#D6336C', '#E8590C']
        kept_ids = {c['label'] for c in clusters}
        traces = []
        for lbl in sorted(set(labels.tolist())):
            m = labels == lbl
            p = pts[m]
            if lbl == -1:
                name, color, size, opacity = 'Noise',                'rgba(180,180,180,0.25)', 1.5, 0.25
            elif lbl in kept_ids:
                name, color, size, opacity = f'Cluster {lbl} (kept)', PALETTE_3D[lbl % len(PALETTE_3D)], 3, 0.9
            else:
                name, color, size, opacity = f'Cluster {lbl} (wall)', 'rgba(90,90,90,0.45)',    2, 0.45
            traces.append(go.Scatter3d(
                x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
                marker=dict(size=size, color=color, opacity=opacity), name=name,
            ))
        fig = go.Figure(traces)
        fig.update_layout(
            title=f'DBSCAN clusters  (eps={DBSCAN_EPS}, minPts={DBSCAN_MIN_SAMPLES}, '
                  f'wall>{WALL_DEPTH_M}m, step={DEPTH_DOWNSAMPLE_STEP})',
            scene=dict(xaxis_title='X (pixel)', yaxis_title='Y (pixel)',
                       zaxis_title='depth (m)',
                       aspectmode='manual', aspectratio=dict(x=1.8, y=1.0, z=0.6)),
            legend=dict(itemsizing='constant'),
            margin=dict(l=0, r=0, b=0, t=40),
        )
        fig.show()
